# 一、前言

光譜數據預處理是光譜分析中的一個重要環節，其目的是通過一系列技術改善數據質量，以提高後續分析的準確性和可靠性。以下是幾個常見的光譜數據預處理步驟：

### 基線校正（Baseline Correction）
去除由於儀器、樣品容器或其他因素引起的基線偏移或漂移，以確保光譜反映的是樣品本身的特性。

### 噪聲降低（Noise Reduction）
利用各種濾波技術如移動平均（Moving Average）、Savitzky-Golay濾波、小波變換（Wavelet Transform）等，以減少光譜數據中的隨機噪聲。

### 歸一化（Normalization）
將光譜的強度比例調整到相同的水平，有助於數據之間的比較，常用的方法包括向量歸一化、最大值歸一化、範數歸一化等。

### 標準正態變量變換（Standard Normal Variate, SNV）
用於矯正散射效應並進行光譜的校正。

### 多元散射校正（Multiplicative Scatter Correction, MSC）
也是一種用於校正樣品散射效應的技術，特別適用於固體和粉末樣品的近紅外（NIR）數據處理。

### 導數光譜（Derivative Spectroscopy）
通過計算一階、二階或更高階導數來突顯光譜曲線的細節，有助於分離重疊峰，並消除基線漂移的影響。

### 去卷積（Deconvolution）
一種數學過程，用於分離光譜中的重疊信號，以獲得更清晰的峰值信息。

這些預處理技術的應用取決於所處理的光譜類型（如UV-Vis、IR、NMR、MS等）、樣品的性質以及分析的目標。預處理步驟的選擇和順序將直接影響光譜分析的效果，因此在實際操作中通常需要結合具體情況進行優化。

# 1. 基線校正
首先，生成一組模擬的光譜數據，然後對數據進行基線扣除。基線是通過選擇非峰區域的數據點並擬合一個二次多項式獲得的。通俗地說，在處理光譜數據時，我們需要確定哪些部分是因為樣品吸收光而產生的信號（即我們感興趣的峰），哪些部分是背景信號或其他原因導致的不相關信號（即基線）。由於這些不相關信號可能會干擾我們對感興趣信號的分析，我們需要將它們去除。

為了做到這一點，我們首先識別出光譜中的峰值區域，然後忽略這些區域，只使用剩下的部分（即非峰值區域的數據點）來估計基線。這些非峰值區域通常認為主要包含背景信號。接下來，我們使用這些選定的數據點來創建一個數學模型，這裡是一個二次多項式，它是一個簡單的彎曲形狀（像一個開口向上或向下的拋物線），這個模型被調整得盡可能地貼近這些選定的數據點。擬合完成後，這個二次多項式就代表了基線。

最後，代碼繪製了兩個子圖，上面的子圖顯示原始數據和計算出的基線，下面的子圖顯示進行了基線校正後的數據。在實際應用中，基線的形狀和數據的複雜性可能會要求使用更高級的基線校正方法。

In [ ]:
import jax.numpy as jnp
from jax import random
import plotly.graph_objects as go
from scipy.signal import find_peaks, peak_widths
import numpy as np

key = random.PRNGKey(0)

x_data = jnp.linspace(400, 700, 300)
y_data = jnp.exp(-0.005 * (x_data - 500) ** 2) + random.normal(key, x_data.shape) * 0.02 + 0.1 * (x_data - 400) ** 2 / 300 ** 2

peaks, _ = find_peaks(y_data, prominence=0.05)
widths = peak_widths(y_data, peaks, rel_height=0.5)

mask = jnp.ones_like(y_data, dtype=bool)

for peak, width in zip(peaks, widths[0]):
    mask = mask.at[int(peak - width):int(peak + width)].set(False)

baseline_values = np.polyfit(np.array(x_data[mask]), np.array(y_data[mask]), 2)
baseline_poly = np.poly1d(baseline_values)
baseline = baseline_poly(np.array(x_data))
y_corrected = y_data - baseline

fig = go.Figure()

fig.add_trace(go.Scatter(x=x_data, y=y_data, mode='lines', name='Original Data'))
fig.add_trace(go.Scatter(x=x_data, y=baseline, mode='lines', name='Baseline', line=dict(dash='dash')))
fig.update_layout(title='Original Spectral Data with Baseline', xaxis_title='Wavelength (nm)', yaxis_title='Intensity')

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=x_data, y=y_corrected, mode='lines', name='Corrected Data', line=dict(color='orange')))
fig2.update_layout(title='Baseline Corrected Data', xaxis_title='Wavelength (nm)', yaxis_title='Intensity')

fig.show()
fig2.show()


In [ ]:
import jax.numpy as jnp
from jax import random
import plotly.graph_objects as go
from scipy.signal import find_peaks, peak_widths
import numpy as np

key = random.PRNGKey(0)

# 生成模擬的光譜數據，包含兩個峰值
x_data = jnp.linspace(400, 700, 300)
y_data = (jnp.exp(-0.005 * (x_data - 500) ** 2) + 
          jnp.exp(-0.005 * (x_data - 600) ** 2) + 
          random.normal(key, x_data.shape) * 0.02 + 
          0.1 * (x_data - 400) ** 2 / 300 ** 2)

# 找到峰值
peaks, _ = find_peaks(y_data, prominence=0.05)
widths = peak_widths(y_data, peaks, rel_height=0.5)

# 創建掩碼，排除峰值區域
mask = jnp.ones_like(y_data, dtype=bool)
for peak, width in zip(peaks, widths[0]):
    mask = mask.at[int(peak - width):int(peak + width)].set(False)

# 擬合二次多項式基線
baseline_values = np.polyfit(np.array(x_data[mask]), np.array(y_data[mask]), 2)
baseline_poly = np.poly1d(baseline_values)
baseline = baseline_poly(np.array(x_data))

# 基線校正
y_corrected = y_data - baseline

# 繪製原始數據和基線
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_data, y=y_data, mode='lines', name='Original Data'))
fig.add_trace(go.Scatter(x=x_data, y=baseline, mode='lines', name='Baseline', line=dict(dash='dash')))
fig.update_layout(title='Original Spectral Data with Baseline', xaxis_title='Wavelength (nm)', yaxis_title='Intensity')

# 繪製基線校正後的數據
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=x_data, y=y_corrected, mode='lines', name='Corrected Data', line=dict(color='orange')))
fig2.update_layout(title='Baseline Corrected Data', xaxis_title='Wavelength (nm)', yaxis_title='Intensity')

fig.show()
fig2.show()

# 2. 噪聲降低

首先，創建了一組模擬的光譜數據，並在其上添加了一定量的隨機噪聲。然後我們使用三種不同的方法來降低噪聲：

- **移動平均（Moving Average）**：通過在一定窗口大小內計算平均值來平滑數據。
- **Savitzky-Golay濾波器（Savitzky-Golay Filter）**：通過在數據上擬合多項式並計算其在每個點的值來平滑數據。
- **小波變換去噪（Wavelet Denoising）**：通過應用小波變換來分離數據中的噪聲和信號，然後去除噪聲。

這裡可能唯一不好理解原理的就是小波變換，這個方法你一定用過多次（本人QQ名都叫Wavelet，因為這個方法用得太多了），但你不一定直觀地理解了它。下面我做一個簡單的解釋：

小波變換是一種工具，它可以將信號拆分成不同頻率的組件，並且還能告訴我們這些組件在時間或空間上的位置。就像一個音樂等化器可以顯示不同音高的音量，並且告訴你音樂的哪一部分有高音或低音一樣。當我們使用小波變換對信號進行分析時，我們基本上是將信號分解成許多小波構件。這些構件在不同的尺度上展示了信號的特性。噪聲通常表現為高頻的部分，因為它們快速變化、無規律，就像是音樂中突兀的刺耳聲。而信號的真實部分通常是更加平滑和有規律的，就像是一致的旋律線。在小波變換後，我們可以通過一個過程叫做閾值處理來移除噪聲。這個過程涉及到設置一個標準（閾值），所有低於這個標準的小波構件（通常是噪聲）都會被視為不重要並被移除或減小它們的強度。這就像是降低音樂等化器上的某些頻率。在去除了噪聲之後，剩下的構件再被組合回去，重建成一個更清晰、噪聲更少的信號。總之，小波變換去噪就像是一個精細的篩子，能夠篩選出真實信號中的好粒子（有用的頻率成分），同時去除掉壞粒子（噪聲），最後我們就能得到一個更加純淨、更接近真實情況的信號。

最後，我們繪製一個包含四個子圖的圖表，分別展示原始數據和三種不同噪聲降低技術的效果。

In [ ]:
import numpy as np

import matplotlib.pyplot as plt

from scipy.signal import savgol_filter

import pywt

np.random.seed(0)  

x_data = np.linspace(400, 700, 300)  

y_data = np.exp(-0.01*(x_data-550)**2) + np.random.normal(0, 0.1, x_data.size)

def moving_average(data, window_size):

    return np.convolve(data, np.ones(window_size)/window_size, mode='same')

def savgol_filtering(data, window_size, order):

    return savgol_filter(data, window_size, order)

def wavelet_denoising(data, wavelet, level):

    coeff = pywt.wavedec(data, wavelet, mode='per', level=level)

    sigma = np.median(np.abs(coeff[-1])) / 0.6745

    uthresh = sigma * np.sqrt(2*np.log(len(data)))

    coeff[1:] = (pywt.threshold(i, value=uthresh, mode='soft') for i in coeff[1:])

    return pywt.waverec(coeff, wavelet, mode='per')

ma_data = moving_average(y_data, 5)  

sg_data = savgol_filtering(y_data, 15, 3)  

wt_data = wavelet_denoising(y_data, 'db1', 1)  

fig, axs = plt.subplots(4, 1, figsize=(10, 12), sharex=True)

axs[0].plot(x_data, y_data, label='Original Data')

axs[0].set_title('Original Spectral Data with Noise')

axs[0].legend()

axs[1].plot(x_data, ma_data, color='green', label='Moving Average')

axs[1].set_title('Moving Average')

axs[1].legend()

axs[2].plot(x_data, sg_data, color='red', label='Savitzky-Golay Filter')

axs[2].set_title('Savitzky-Golay Filter')

axs[2].legend()

axs[3].plot(x_data, wt_data, color='purple', label='Wavelet Denoising')

axs[3].set_title('Wavelet Denoising')

axs[3].legend()

for ax in axs:

    ax.set_xlabel('Wavelength (nm)')

    ax.set_ylabel('Intensity')

plt.tight_layout()

plt.show()

# 3. 歸一化

光譜數據歸一化是處理光譜數據的常見步驟，其目的是將數據的範圍調整到 [0, 1] 或 [-1, 1] 等標準區間內。原始數據和歸一化結果之間在視覺上可能沒有明顯區別，這是因為歸一化過程僅僅改變了數據的數值範圍，並沒有改變數據的整體形狀或者模式。

In [ ]:
import numpy as np

import matplotlib.pyplot as plt

np.random.seed(0)

x_data = np.linspace(400, 700, 300)

y_data = (np.exp(-0.01*(x_data - 550)**2) +

          np.random.normal(0, 0.1, x_data.size)) * 1000

def normalize_data(data):

    min_val = np.min(data)

    max_val = np.max(data)

    norm_data = (data - min_val) / (max_val - min_val)

    return norm_data

normalized_y_data = normalize_data(y_data)

fig, axs = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

axs[0].plot(x_data, y_data, label='Original Data')

axs[0].set_title('Original Spectral Data')

axs[0].set_ylabel('Intensity (Arbitrary Units)')  

axs[0].legend()

axs[1].plot(x_data, normalized_y_data, color='red', label='Normalized Data')

axs[1].set_title('Normalized Spectral Data')

axs[1].set_ylabel('Normalized Intensity')  

axs[1].legend()

for ax in axs:

    ax.set_xlabel('Wavelength (nm)')

plt.tight_layout()

plt.show()

# 4. 標準正態變量變換

標準正態變量變換，也稱為 Z 分數標準化，是指將數據點轉換為其與均值的距離除以標準差的值。這種轉換後的數據的均值為 0，標準差為 1，遵循標準正態分布。

In [ ]:
np.random.seed(0)

x_data = np.linspace(400, 700, 300)

y_data = np.exp(-0.01*(x_data - 550)**2) + np.random.normal(0, 0.1, x_data.size)

def standardize_data(data):

    mean_val = np.mean(data)

    std_dev = np.std(data)

    standardized_data = (data - mean_val) / std_dev

    return standardized_data

standardized_y_data = standardize_data(y_data)

fig, axs = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

axs[0].plot(x_data, y_data, label='Original Data')

axs[0].set_title('Original Spectral Data')

axs[0].set_ylabel('Intensity')

axs[0].legend()

axs[1].plot(x_data, standardized_y_data, color='red', label='Standardized Data')

axs[1].set_title('Standardized Spectral Data')

axs[1].set_ylabel('Z-Score')

axs[1].legend()

for ax in axs:

    ax.set_xlabel('Wavelength (nm)')

plt.tight_layout()

plt.show()

# 5. 多元散射校正

多元散射校正（Multiple Scatter Correction, MSC）是一種用於校正光譜數據中散射影響的技術，通常用於近紅外（NIR）光譜分析中。MSC 的基本思想是通過校正樣品光譜，使其接近於一個參考光譜，通常是均值光譜或者一個選定的標準光譜。

通俗解釋其數學原理，可以分為以下四個步驟：

1. **均值光譜的計算**：首先，從一系列光譜數據中計算出一個平均光譜。這個均值光譜可以被看作是包含了所有樣本共有特徵的一個代表。

2. **線性關係的建立**：MSC 處理的關鍵在於假設每個樣本的光譜可以通過一個線性變換來逼近這個均值光譜。線性變換指的是可以通過伸縮（乘以一個係數）和平移（加或減去一個常數）操作來調整一個樣本光譜，使其形狀盡可能接近平均光譜。

3. **線性回歸擬合**：對於每個樣本的光譜，我們通過線性回歸找到最佳的伸縮係數和平移常數。這個過程類似於在散點圖中找到一條最佳直線，使得所有點到這條線的距離之和最小。在 MSC 中，"點"就是樣本光譜的每一個數據點，而"直線"則是我們通過伸縮和平移操作嘗試匹配的均值光譜。

4. **校正應用**：一旦我們找到了最佳的伸縮係數和平移常數，我們就將這些參數應用於原樣本光譜，進行調整。這樣，原樣本光譜就被校正為一個新的光譜，理論上應該減少了由於散射造成的變異，而更能反映樣本的真實化學信息。

In [ ]:
import numpy as np

import matplotlib.pyplot as plt

def generate_spectral_data(num_samples, num_features):

    np.random.seed(0)  

    baseline = np.outer(np.linspace(1, 2, num_samples), np.linspace(0, 1, num_features))

    peaks = np.exp(

        -((np.linspace(0, num_features, num_features) - num_features / 3) ** 2) / (2 * (num_features / 20) ** 2))

    peaks = np.outer(np.random.rand(num_samples), peaks) * 50

    scatter = np.outer(0.5 * (np.random.rand(num_samples) - 0.5) * np.linspace(0, 10, num_samples),

                       np.ones(num_features))

    spectral_data = baseline + peaks + scatter

    return spectral_data

def do_msc(input_data):

    ref_spectrum = np.mean(input_data, axis=0)

    corrected_data = np.zeros_like(input_data)

    for i in range(input_data.shape[0]):

        fit = np.polyfit(ref_spectrum, input_data[i, :], 1)

        corrected_data[i, :] = (input_data[i, :] - fit[1]) / fit[0]

    return corrected_data, ref_spectrum

num_samples = 10

num_features = 300

spectra = generate_spectral_data(num_samples, num_features)

corrected_spectra, mean_spectrum = do_msc(spectra)

fig, ax = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

for i in range(num_samples):

    ax[0].plot(spectra[i, :], label=f'Sample {i + 1}' if i == 0 else None)

ax[0].set_title('Original Spectra')

ax[0].set_ylabel('Intensity')

ax[0].legend()

for i in range(num_samples):

    ax[1].plot(corrected_spectra[i, :], label=f'Sample {i + 1}' if i == 0 else None)

ax[1].set_title('MSC Corrected Spectra')

ax[1].set_ylabel('Intensity')

ax[1].set_xlabel('Wavelength Index')

ax[1].legend()

plt.tight_layout()

plt.show()

# 6. 一階導數光譜

一階導數光譜的數學原理基本上和任何函數的一階導數原理是相同的。在數學中，一階導數描述了一個函數在某一點上的斜率，也就是說，它可以告訴我們這個函數在這一點上是怎樣增加或減少的。

將這個概念應用到光譜分析上，一個光譜通常是將吸收或發射的光強度與波長（或頻率）的關係圖表。當我們對這個關係圖表進行一階導數處理時，我們得到的新曲線不再直接顯示光強度隨波長的變化，而是顯示光強度變化速率隨波長的變化。換句話說，這個導數光譜告訴我們隨著波長的增加，光強度是如何快速增加或減少的。

這在分析光譜時是非常有用的，因為它可以幫助我們識別光譜中的細微特徵，這些細微特徵在原始光譜中可能不是很明顯。比如，一階導數光譜可以更清晰地顯示出光譜中的峰值和谷值，因為它強調了光強度變化的地方（即原始光譜的斜率最大的地方）。這些峰值和谷值通常對應於特定的物質吸收或發射光的特徵波長，因此可以用於識別和分析這些物質。

In [ ]:
import numpy as np

import matplotlib.pyplot as plt

def generate_spectral_data(num_samples, num_features):

    x = np.linspace(400, 700, num_features)  

    spectra = np.zeros((num_samples, num_features))

    for i in range(num_samples):

        baseline = np.polyval([1e-4, -0.1, 10], x) * (0.5 + np.random.rand() / 2)

        peak = np.exp(-((x - 500 - np.random.rand() * 100) ** 2) / (2 * (25 ** 2)))

        spectra[i, :] = baseline + peak * np.random.rand() * 100

    return x, spectra

def calculate_derivative(x, spectra):

    dx = np.diff(x)

    derivative_spectra = np.diff(spectra) / dx[0]

    x_mid_points = (x[:-1] + x[1:]) / 2

    return x_mid_points, derivative_spectra

num_samples = 5

num_features = 1000

x, spectra = generate_spectral_data(num_samples, num_features)

x_deriv, derivative_spectra = calculate_derivative(x, spectra)

fig, axes = plt.subplots(2, 1, figsize=(8, 10), sharex=True)

for i in range(num_samples):

    axes[0].plot(x, spectra[i, :], label=f'Sample {i + 1}')

axes[0].set_title('Original Spectra')

axes[0].set_ylabel('Intensity')

axes[0].legend()

for i in range(num_samples):

    axes[1].plot(x_deriv, derivative_spectra[i, :], label=f'Sample {i + 1}')

axes[1].set_title('First Derivative of Spectra')

axes[1].set_xlabel('Wavelength (nm)')

axes[1].set_ylabel('First Derivative Intensity')

axes[1].legend()

plt.tight_layout()

plt.show()

# Python 多峰高斯函數擬合的步驟指南

在數據分析和機器學習領域，多峰高斯函數擬合是一種用來模擬複雜分布的強大工具。對於初學者來說，理解這個過程的每一步至關重要。本文將逐步指導你如何在 Python 中實現多峰高斯函數擬合，包括必要的代碼和註釋，以幫助你更好地理解整個過程。

## 整體流程

在進行多峰高斯函數擬合時，我們可以遵循以下步驟：

1. **準備數據**：首先，我們需要準備好要進行擬合的數據。這些數據通常來自實驗測量或模擬計算。

2. **定義高斯函數**：接下來，我們需要定義一個或多個高斯函數。高斯函數是一種常見的數學函數，用於描述數據中的峰值。

3. **擬合數據**：使用適當的數學工具和算法，我們將高斯函數擬合到數據上。這一步通常涉及最小化誤差函數，以找到最佳的高斯函數參數。

4. **結果可視化**：最後，我們將擬合結果可視化，以便更好地理解和解釋數據。這通常包括繪製原始數據和擬合曲線。


# 步驟1：準備數據
首先，我們需要一個數據集來進行高斯擬合。可以使用NumPy庫來生成一些具有多峰的模擬數據。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 设定随机种子以便重复实验
np.random.seed(0)

# 生成x数据
x = np.linspace(-10, 10, 1000)

# 生成多峰高斯分布的数据
# 这里我们假设有两个高斯峰
y = (0.5 * np.exp(-(x-1)**2 / (2*0.5**2)) +
     2 * np.exp(-(x+1)**2 / (2*0.5**2))) + 0.1 * np.random.normal(size=x.size)

# 绘制数据
plt.plot(x, y, label='Data with multiple peaks')
plt.title('Simulated Multi-Peak Gaussian Data')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.legend()
plt.show()

def gaussian(x, amplitude, mean, std_dev):
    """定义高斯函数"""
    return amplitude * np.exp(-((x - mean) ** 2) / (2 * std_dev ** 2))

from scipy.optimize import curve_fit

def multi_gaussian(x, *params):
    """多高斯函数，可以接收任意个高斯函数"""
    # params包含了多个高斯函数的参数
    total = np.zeros_like(x)
    for i in range(0, len(params), 3):
        amplitude = params[i]
        mean = params[i+1]
        std_dev = params[i+2]
        total += gaussian(x, amplitude, mean, std_dev)
    return total

# 初始参数：振幅、均值和标准差
initial_params = [1.0, 3.0, 0.5, 1.0, -3.0, 0.5]  # 两个高斯函数

# 数据拟合
popt, pcov = curve_fit(multi_gaussian, x, y, p0=initial_params)

print("拟合的参数:", popt)

# 绘制原始数据
plt.plot(x, y, label='Data with multiple peaks')

# 计算拟合值
fitted_data = multi_gaussian(x, *popt)

# 绘制拟合结果
plt.plot(x, fitted_data, label='Fitted multi-Gaussian', color='red')
plt.legend()
plt.title('Multi-Peak Gaussian Fitting Result')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# 设定随机种子以便重复实验
np.random.seed(0)

# 生成x数据
x = np.linspace(-10, 10, 1000)

# 生成多峰高斯分布的数据
# 这里我们假设有两个高斯峰
y = (0.5 * np.exp(-(x-1)**2 / (2*0.5**2)) +
     2 * np.exp(-(x+1)**2 / (2*0.5**2))) + 0.1 * np.random.normal(size=x.size)

# 绘制数据
plt.plot(x, y, label='Data with multiple peaks')
plt.title('Simulated Multi-Peak Gaussian Data')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.legend()
plt.show()

def gaussian(x, amplitude, mean, std_dev):
    """定义高斯函数"""
    return amplitude * np.exp(-((x - mean) ** 2) / (2 * std_dev ** 2))

def multi_gaussian(x, *params):
    """多高斯函数，可以接收任意个高斯函数"""
    total = np.zeros_like(x)
    for i in range(0, len(params), 3):
        amplitude = params[i]
        mean = params[i+1]
        std_dev = params[i+2]
        total += gaussian(x, amplitude, mean, std_dev)
    return total

# 初始参数：振幅、均值和标准差
initial_params = [0.5, 1.0, 0.5, 2.0, -1.0, 0.5]  # 两个高斯函数

# 数据拟合
popt, pcov = curve_fit(multi_gaussian, x, y, p0=initial_params)

print("拟合的参数:", popt)

# 绘制原始数据
plt.plot(x, y, label='Data with multiple peaks')

# 计算拟合值
fitted_data = multi_gaussian(x, *popt)

# 绘制拟合结果
plt.plot(x, fitted_data, label='Fitted multi-Gaussian', color='red')
plt.legend()
plt.title('Multi-Peak Gaussian Fitting Result')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# 设定随机种子以便重复实验
np.random.seed(0)

# 生成x数据
x = np.linspace(-10, 10, 1000)

# 生成多峰高斯分布的数据
# 这里我们假设有两个高斯峰
y = (0.5 * np.exp(-(x-1)**2 / (2*0.5**2)) +
     2 * np.exp(-(x+1)**2 / (2*0.5**2))) + 0.1 * np.random.normal(size=x.size)

# 绘制数据
plt.plot(x, y, label='Data with multiple peaks')
plt.title('Simulated Multi-Peak Gaussian Data')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.legend()
plt.show()

def gaussian(x, amplitude, mean, std_dev):
    """定义高斯函数"""
    return amplitude * np.exp(-((x - mean) ** 2) / (2 * std_dev ** 2))

def fit_single_gaussian(x, y, initial_params):
    """拟合单个高斯峰"""
    popt, _ = curve_fit(gaussian, x, y, p0=initial_params)
    return popt

# 初始参数：振幅、均值和标准差
initial_params_list = [
    [0.5, 1.0, 0.5],  # 第一个高斯函数的初始参数
    [2.0, -1.0, 0.5]  # 第二个高斯函数的初始参数
]

fitted_params = []
y_fit = np.zeros_like(y)

for initial_params in initial_params_list:
    popt = fit_single_gaussian(x, y - y_fit, initial_params)
    fitted_params.extend(popt)
    y_fit += gaussian(x, *popt)

print("拟合的参数:", fitted_params)

# 绘制原始数据
plt.plot(x, y, label='Data with multiple peaks')

# 绘制拟合结果
plt.plot(x, y_fit, label='Fitted multi-Gaussian', color='red')
plt.legend()
plt.title('Multi-Peak Gaussian Fitting Result')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# 设定随机种子以便重复实验
np.random.seed(0)

# 生成x数据
x = np.linspace(-10, 10, 1000)

# 生成多峰高斯分布的数据
# 这里我们假设有两个高斯峰
y = (0.5 * np.exp(-(x-1)**2 / (2*0.5**2)) +
     2 * np.exp(-(x+1)**2 / (2*0.5**2))) + 0.1 * np.random.normal(size=x.size)

# 绘制数据
plt.plot(x, y, label='Data with multiple peaks')
plt.title('Simulated Multi-Peak Gaussian Data')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.legend()
plt.show()

def gaussian(x, amplitude, mean, std_dev):
    """定义高斯函数"""
    return amplitude * np.exp(-((x - mean) ** 2) / (2 * std_dev ** 2))

def fit_single_gaussian(x, y, initial_params):
    """拟合单个高斯峰"""
    popt, _ = curve_fit(gaussian, x, y, p0=initial_params, maxfev=10000)
    return popt

def calculate_r_squared(y, y_fit):
    """计算R-squared值"""
    ss_res = np.sum((y - y_fit) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    return 1 - (ss_res / ss_tot)

# 初始参数：振幅、均值和标准差
initial_params_list = [
    [0.5, 1.0, 0.5],  # 第一个高斯函数的初始参数
    [2.0, -1.0, 0.5]  # 第二个高斯函数的初始参数
]

fitted_params = []
y_fit = np.zeros_like(y)
r_squared = 0
r_squared_threshold = 1e-6  # R-squared变化的阈值
max_iterations = 10  # 最大迭代次数
iteration = 0

while iteration < max_iterations:
    iteration += 1
    for initial_params in initial_params_list:
        popt = fit_single_gaussian(x, y - y_fit, initial_params)
        fitted_params.extend(popt)
        y_fit += gaussian(x, *popt)
    
    new_r_squared = calculate_r_squared(y, y_fit)
    if abs(new_r_squared - r_squared) < r_squared_threshold:
        break
    r_squared = new_r_squared

print("拟合的参数:", fitted_params)
print("最终的R-squared值:", r_squared)

# 绘制原始数据
plt.plot(x, y, label='Data with multiple peaks')

# 绘制拟合结果
plt.plot(x, y_fit, label='Fitted multi-Gaussian', color='red')
plt.legend()
plt.title('Multi-Peak Gaussian Fitting Result')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.show()

In [ ]:
import jax.numpy as jnp
from jax import random
import plotly.graph_objects as go
from scipy.optimize import curve_fit
from scipy.signal import find_peaks

# 设定随机种子以便重复实验
key = random.PRNGKey(0)

# 生成x数据
x = jnp.linspace(-10, 10, 1000)

# 生成多峰高斯分布的数据
# 这里我们假设有两个高斯峰
y = (0.6 * jnp.exp(-(x-2.1)**2 / (2*0.5**2)) +
    10 * jnp.exp(-(x+1)**2 / (2*0.5**2))) + 0.1 * random.normal(key, shape=x.shape)

# 绘制数据
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name='Data with multiple peaks'))

def gaussian(x, amplitude, mean, std_dev):
    """定義高斯函數"""
    return amplitude * jnp.exp(-((x - mean) ** 2) / (2 * std_dev ** 2))

def multi_gaussian(x, *params):
    """多高斯函数，可以接收任意个高斯函数"""
    total = jnp.zeros_like(x)
    for i in range(0, len(params), 3):
       amplitude = params[i]
       mean = params[i+1]
       std_dev = params[i+2]
       total += gaussian(x, amplitude, mean, std_dev)
    return total

def calculate_r_squared(y, y_fit):
    """計算R-squared值"""
    ss_res = jnp.sum((y - y_fit) ** 2)
    ss_tot = jnp.sum((y - jnp.mean(y)) ** 2)
    return 1 - (ss_res / ss_tot)

def fit_multi_gaussian(x, y, initial_params, r_squared_threshold=1e-4, max_iterations=100, min_iterations=2):
    r_squared = 0
    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        popt, _ = curve_fit(multi_gaussian, x, y, p0=initial_params)
        y_fit = multi_gaussian(x, *popt)
        
        new_r_squared = calculate_r_squared(y, y_fit)
        print(f"Iteration {iteration}: R-squared = {new_r_squared}, Parameters = {popt}")
        if iteration >= min_iterations and abs(new_r_squared - r_squared) < r_squared_threshold:
            break
        r_squared = new_r_squared

    return popt, y_fit, r_squared

# 初始参数：振幅、均值和标准差
initial_params = [0.5, 1.0, 0.5, 2.0, -1.0, 0.5]  # 两个高斯函数

# 调用函数进行拟合
popt, y_fit, r_squared = fit_multi_gaussian(x, y, initial_params)

# 找到拟合曲线中的峰值位置
peaks, _ = find_peaks(y_fit)

# 绘制拟合结果
fig.add_trace(go.Scatter(x=x, y=y_fit, mode='lines', name='Fitted multi-Gaussian', line=dict(color='red')))

# 标注峰值位置
fig.add_trace(go.Scatter(x=x[peaks], y=y_fit[peaks], mode='markers', name='Peaks', marker=dict(color='blue', symbol='x')))
for peak in peaks:
    fig.add_annotation(x=x[peak], y=y_fit[peak], text=f'{x[peak]:.2f}', showarrow=True, arrowhead=2)

# 绘制未叠加的两个高斯拟合
for i in range(0, len(popt), 3):
    amplitude = popt[i]
    mean = popt[i+1]
    std_dev = popt[i+2]
    single_gaussian = gaussian(x, amplitude, mean, std_dev)
    fig.add_trace(go.Scatter(x=x, y=single_gaussian, mode='lines', name=f'Gaussian {i//3 + 1}', line=dict(dash='dash')))

fig.update_layout(title='Multi-Peak Gaussian Fitting Result', xaxis_title='X-axis', yaxis_title='Y-axis')
fig.show()


代碼解釋：

- `np.linspace(-10, 10, 1000)` 生成了從 -10 到 10 的 1000 個點。
- `np.exp` 計算高斯函數的值。
- `np.random.normal` 在生成的數據上添加一些噪聲以使其更真實。

# 步驟2：定義高斯函數
接下來，我們需要定義一個高斯函數，以便於擬合。

In [10]:
def gaussian(x, amplitude, mean, std_dev):
    """定义高斯函数"""
    return amplitude * np.exp(-((x - mean) ** 2) / (2 * std_dev ** 2))


代码解释：

gaussian 函數用於計算給定參數的高斯函數值，參數包括振幅、均值和標準差。

# 步驟3：擬合數據
然後，我們使用 `scipy` 庫的 `curve_fit` 方法來擬合數據。我們將在這裡同時考慮多個高斯函數。

In [ ]:
from scipy.optimize import curve_fit

def multi_gaussian(x, *params):
    """多高斯函数，可以接收任意个高斯函数"""
    # params包含了多个高斯函数的参数
    total = np.zeros_like(x)
    for i in range(0, len(params), 3):
        amplitude = params[i]
        mean = params[i+1]
        std_dev = params[i+2]
        total += gaussian(x, amplitude, mean, std_dev)
    return total

# 初始参数：振幅、均值和标准差
initial_params = [1.0, 3.0, 0.5, 1.0, -3.0, 0.5]  # 两个高斯函数

# 数据拟合
popt, pcov = curve_fit(multi_gaussian, x, y, p0=initial_params)

print("拟合的参数:", popt)

代码解释：

multi_gaussian 函数处理多个高斯项，提取振幅、均值和标准差。
curve_fit用于进行数据拟合，p0设定了初始参数。

# 步驟4：結果可視化
最後，我們將輸出結果並繪製擬合曲線與原始數據進行比較。


In [ ]:
# 绘制原始数据
plt.plot(x, y, label='Data with multiple peaks')

# 计算拟合值
fitted_data = multi_gaussian(x, *popt)

# 绘制拟合结果
plt.plot(x, fitted_data, label='Fitted multi-Gaussian', color='red')
plt.legend()
plt.title('Multi-Peak Gaussian Fitting Result')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.show()

代码解释：

multi_gaussian(x, *popt)使用拟合的参数生成拟合结果。
两条曲线同时绘制，便于直观比较。

结论
通过上述步骤，我们成功实现了Python中多峰高斯函数的拟合。这一过程包括准备数据、定义高斯函数、使用拟合算法以及结果的可视化。熟悉这几个步骤后，你就可以在真正的数据集上运用这个技巧了。

独立探索是学习编程的一个重要部分。在你掌握了基础后，不妨尝试将所学的内容应用于不同的数据集，看看结果如何变化。希望这篇文章能为你的学习之旅提供一些有用的指导。如果在实现过程中遇到任何问题，随时欢迎讨论！